## Module 8, Lecture 1 - Exploratory Data Analysis

Any statistical modeling project (or any data analysis project, for that matter) must start with Exploratory Data Analysis (or EDA). Recall from your 'Tidyverse Skills for Data Science' textbook that there are 7 steps in a data science project:  

• Define the question you want to ask the data  
• Get the data  
• Clean the data  
• Explore the data  
• Fit statistical models  
• Communicate the results  
• Make your analysis reproducible  

We have already covered parts 2 and 3 - now we will focus on part 4, Explore the Data.  
### Exploratory Data Analysis (EDA)
So what do we mean by EDA? According to our 'Tidyverse Skills for Data Science' textbook, Descriptive and Exploratory analyses develop simple summaries about the data you're working with and how the variables might relate to each other. This step can help to discover new questions about the data, it can help to spot any data issues (e.g., irregularities, anomalies, outliers). It can also help us think about whether any transformations or normalizations might be needed. It can also help us think about what kind of statistical model we may want to fit.

The use of descriptive statistics, such as measures of central tendency (i.e., mean, median, mode) and measures of variability (i.e., range, standard deviations, or variance), are all critical parts of an EDA. 

### Libraries needed 
We will need tidyverse and ggplot2


In [ ]:
library(tidyverse)
library(ggplot2)

### Reading in the Data
This dataset comes from Sommer et al. (2018), "Comparing apples to apples: an environmental criminology analysis of the effects of heat and rain on violent crimes in Boston."  

In [ ]:
data <- read_csv("https://raw.githubusercontent.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/refs/heads/main/course-3-data-cleaning-and-analysis/module-8/data/weather_crimes_Boston.csv")

An overview of what these variables stand for (from what I could surmise, as no metadata were included with the data files)
  - Date: date in YYYY-MM-DD  
  - Day_Wk: day of the week  
  - nb_crimes: number of total crimes  
  - nb... : number of type of crimes  
  - TEMP: temperature in C  
  - TMAX_C: maximum temperature in C  
  - TMIN_C: minimum temperature in C  
  - DEWP: dewpoint  
  - VISIB: visibility  
  - WIND: wind speed  
  - PRCP: precipitation  
  - y20XX: year of data (0/1)  
  - year: year of data  
  - Friday...Weekend: day of week of data (0/1)  
  - _lag: series of lag variables  
  - Season: season of date  
  - Summer...Spring: Season (0/1)  
  - heat.index: heat index  
  - SNOW: whether there was recorded SNOW  

Let's first use our trusty functions `glimpse` etc. to take a look at what's in the data:

In [ ]:
glimpse(data)

### Exploring Missingness in the Data 
As the 'Tidyverse' textbook describes, missing data can cause a whole host of problems in data analysis. You've already experienced some of these challenges, for example, the need to specify `na.rm=TRUE` so that you can calculate summary statistics like `mean`. Therefore, it's best to get an understanding of missingness in your data from the get-go. In R, we usually specify a missing value with "NA", but as we've seen in the real world, sometimes different data providers/authors will code missingness in different ways - with blank cells, N/A, na, -999, etc. 
From when you used the `glimpse` function above, what variables can you see have missing values? How can you tell?

## IN CLASS: We can also use our tidyverse functions `summarise_all`:

In [ ]:
data %>% summarise_all(funs(sum(is.na(.))))
data %>% summarise(across(everything(),~ sum(is.na(.))))

## IN CLASS: How might you calculate the proportion of missingness for each variable?

In [ ]:
data %>% select_if(is.numeric) %>% summarise_all(funs(mean(is.na(.))))

data %>% summarise(across(everything(),~ mean(is.na(.))))

There are also a few packages that have been developed to visualize missingness:

In [ ]:
install.packages("naniar")
library(naniar)

# visualize missingness
vis_miss(data)

Wow! It looks like our data are pretty complete. How can you tell?

*Work up to here and we will pick up from here for Class*

### Describing the Data's Shape
Determining the 'shape' of your data or a variable in question is essential because statistical methods used for inference often require your data to be distributed in a certain manner (i.e., normal distribution) before they can be applied to the data.   

What do we mean by shape? You'll want to consider how the observations within the variable's range are distributed - are they clustered towards the middle, are they skewed in some way, are the spread out or clustered together? 

How can we describe a data's shape? **Hint: histograms! Plot the distribution of `nb_crimes`:

In [ ]:
data %>% 
  ggplot(aes(x=nb_crimes)) +
  geom_density() +
  theme_classic()

How would you describe the shape of `nb_crimes` distribution? Is this variable skewed? Do you see any outliers? 

What about for some other crime types? Make a boxplot that shows the distributions of all of the crime subtypes. Hint: don't forget to `pivot_longer`!


In [ ]:
data %>% pivot_longer(c(nb_aggravated:nb_fraud), names_to="crime_type", values_to="value") %>%
  ggplot(aes(x=crime_type, y=value, fill=crime_type)) +
  geom_boxplot()+
  theme_classic()

What does this tell us that we didn't see from the `nb_crimes` distribution? Are there any outliers that you observe in any of the crime types? 

By default, boxplots flag "outliers" as any observations beyond 1.5 times the interquartile range (IQR or the length of the box), which is the distance between the first and third quartiles. This provides a mathematical way to identify potential outliers, making them easy to spot visually. However, it's ultimately up to you, the analyst, to decide whether an observation should be removed.

In [ ]:
data %>% ggplot(aes(x=nb_robbery)) +
  geom_density()+
  theme_classic()

### Calculating summary statistics
You will want to calculate summary statistics for your data that give you a sense of the central tendency (e.g., mean, median, mode), as well as the variability in the data (e.g., variance and standard deviation).

We've already practiced how to calculate some of these measures with `mean` and `sd` using the `summarise` function. Try calculating the mean and sd for a few of the crime types (i.e., nb_aggravated or nb_robbery) - on average, which types of crimes are more common?

In [ ]:
data %>% dplyr::summarise(mean=mean(nb_aggravated), sd=sd(nb_aggravated))

Now try using the `summary` base R function:

In [ ]:
summary(data)

We can also use the `skim` function in the `skimr` package for a quick summary:

In [ ]:
install.packages("skimr")
options(scipen=99999)
library(skimr)
skimmed <- skim(data)
print(skimmed, strip_metadata = FALSE)

# Viewing the summary statistics in a table 
data %>% select_if(is.numeric) %>% skim() %>% as_tibble()

skim(data) %>% filter(skim_variable == "nb_crimes")

### Evaluating relationships between variables
You may also want to take a look at relationships between variables. You could individually create bivariate scatterplots to see whether there are any correlations (remember - the correlation coefficient ranges between -1 and 1, with -1 meaning a perfectly negative correlation with 1 meaning a perfectly positive correlation) in your continuous variables. 

Pick two variables to plot using ggplot in a scatterplot:

In [ ]:
ggplot(data, aes(x=TEMP, y=nb_crimes)) +
  geom_point() +
  theme_bw()

What do you think? Is there enough of a correlation to think that there might be a relationship between the two variables? What are ways we might be able to evaluate the strength of this relationship?

#### Using GGally for Quick Correlational Analysis
A package I like to use is GGally, which has a few built-in functions to explore your data. Since our data have many variables, let's just select a few to consider. The first function is `ggcorr`, which will plot a heat map that will allow us to see which variables might be correlated.

In [ ]:
names(data)
install.packages("GGally")
library(GGally)
data %>% dplyr::select(nb_crimes:nb_fraud, TEMP:Wind) %>%
  ggcorr()

Question: what does this plot tell you about the variables we selected? What insights can you glean here?

The `GGally` library also has a scatterplot matrix function called `ggpairs` that allows us to view many bivariate scatterplots at once. This is an important function because it will help us identify variables that we may want to include in a statistical inference model later. Since there are 

In [ ]:
data %>% dplyr::select(nb_crimes:nb_fraud, TEMP:Wind) %>% ggpairs() # if you get an error trying to run this in the notebook itself, try pasting the code directly into the console.

Question: which variables are highly correlated? Which ones are not? How do we measure the strength of the correlation? 

## IN CLASS Practice EDA Questions
Exploring larceny crimes - is this type of crime truly much more common than other types, or are there outliers that we should consider removing?

In [ ]:
data_long <- data %>% pivot_longer(c(nb_aggravated:nb_fraud), names_to="crime_type", 
             values_to="value") %>%
             dplyr::select(crime_type, value, Date, Season, TEMP:PRCP) 

What might explain the outliers in the larceny crimes? What about seasonality?

In [ ]:
# plot a histogram that can helps us focus in on some of the outliers. Perhaps there's a seasonal difference? How could we tell?
data_long %>% filter(crime_type == "nb_larceny") %>%
  ggplot(aes(x=value, fill=as.factor(Season))) +
  geom_histogram()

It's clear that the outlier - a day with more than 60 larceny crimes (ie theft of personal property) falls in Summer and then compared to Winter, there are far fewer larcenies occurring. 

How could we evaluate whether these differences are actually significant?

In [ ]:
# use a linear regression 
model <- lm(value ~ Season, data = data_long %>% filter(crime_type=="nb_larceny"))
summary(model)

# basically same as a Chi-square test or looking at differences between group means
# Chi-square test:
df_larceny <- data_long %>% filter(crime_type == "nb_larceny")

# Create a contingency table of season vs number of larceny crimes
larceny_table <- table(df_larceny$Season)

# Perform a chi-square test
chisq.test(larceny_table) # There is a statistically significant difference in the number of larceny crimes across seasons.This means the number of larceny crimes varies by season in a way that is unlikely due to random chance.

Ask students whether they might have any other hypotheses:
- perhaps one of the weather variables has more of an effect than others?
- perhaps some crime types are more sensitive to weather effects than others?
- perhaps holidays or days of the week (weekends vs. weekdays) might have an effect
- perhaps a given year was worse than another?
- what about interaction terms? There are a few included in the data frame (WindxRain and WindxHheatIndex). What about PRCP and Heat? Have them create this interaction term.

Have them explore some of these questions in their groups.